# Multilingual Scientific Translation Chatbot with RAG

**Author:** Sharadananda Mondal  
**Date:** March 2026  
**GitHub:** github.com/sharadananda

## Overview

This notebook implements a multilingual translation chatbot designed for **scientific document translation** across major European languages. It incorporates **Retrieval-Augmented Generation (RAG)** to inject domain-specific terminology context at inference time, improving translation accuracy for specialised scientific content.

The project is motivated by a core limitation of current machine translation systems: **fluency does not equal accuracy in specialised domains**. General-purpose MT models trained on broad corpora consistently fail on low-frequency scientific terminology, producing fluent but semantically incorrect translations. RAG-based context injection at inference time offers a promising route to address this without full model retraining.

### Languages Supported
English, German, French, Spanish, Italian, Portuguese, Dutch, Polish

### Architecture
- **Translation models:** Helsinki-NLP OPUS-MT models (Hugging Face) — free, open-source, state-of-the-art for European language pairs
- **RAG component:** TF-IDF-based retrieval over a scientific glossary; retrieved definitions injected as context before translation
- **Evaluation:** BLEU score (Papineni et al., 2002) and qualitative analysis of domain-specific term handling

### Research Context
This work is directly relevant to the CHIST-ERA project **TINE (Translation is Not Enough)**, which aims to improve machine translation of complex scientific documents across Europe, with accessibility measures including text simplification and terminology explanation.

### Key References
- Vaswani et al. (2017). *Attention Is All You Need.* NeurIPS.
- Bahdanau et al. (2015). *Neural Machine Translation by Jointly Learning to Align and Translate.* ICLR.
- Lewis et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS.
- Papineni et al. (2002). *BLEU: a Method for Automatic Evaluation of Machine Translation.* ACL.

## 1. Installation

Run this cell once to install required packages.

In [1]:
# Install required packages
import sys
!{sys.executable} -m pip install transformers sentencepiece sacrebleu scikit-learn torch ipywidgets --quiet
print("All packages installed successfully.")

All packages installed successfully.


## 2. Imports & Configuration

In [2]:
import warnings
warnings.filterwarnings('ignore')

from transformers import MarianMTModel, MarianTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sacrebleu
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import numpy as np
import re

print("Imports successful.")

Imports successful.


## 3. Language Configuration

We use **Helsinki-NLP OPUS-MT** models from Hugging Face. These are neural MT models trained on the OPUS corpus — one of the largest open-source parallel corpora for European languages. Each language pair uses a dedicated model fine-tuned specifically for that pair, giving significantly better quality than a single multilingual model.

Language codes follow ISO 639-1 standard.

In [3]:
# Supported languages: display name -> ISO code
LANGUAGES = {
    "English":    "en",
    "German":     "de",
    "French":     "fr",
    "Spanish":    "es",
    "Italian":    "it",
    "Portuguese": "pt",
    "Dutch":      "nl",
    "Polish":     "pl"
}

# Helsinki-NLP model naming: opus-mt-{src}-{tgt}
# Some pairs use grouped codes (e.g. 'ROMANCE' for Romance languages)
# We define explicit model mappings for best quality
MODEL_MAP = {
    # From English
    ("en", "de"): "Helsinki-NLP/opus-mt-en-de",
    ("en", "fr"): "Helsinki-NLP/opus-mt-en-fr",
    ("en", "es"): "Helsinki-NLP/opus-mt-en-es",
    ("en", "it"): "Helsinki-NLP/opus-mt-en-it",
    ("en", "pt"): "Helsinki-NLP/opus-mt-en-pt",
    ("en", "nl"): "Helsinki-NLP/opus-mt-en-nl",
    ("en", "pl"): "Helsinki-NLP/opus-mt-en-pl",
    # To English
    ("de", "en"): "Helsinki-NLP/opus-mt-de-en",
    ("fr", "en"): "Helsinki-NLP/opus-mt-fr-en",
    ("es", "en"): "Helsinki-NLP/opus-mt-es-en",
    ("it", "en"): "Helsinki-NLP/opus-mt-it-en",
    ("pt", "en"): "Helsinki-NLP/opus-mt-pt-en",
    ("nl", "en"): "Helsinki-NLP/opus-mt-nl-en",
    ("pl", "en"): "Helsinki-NLP/opus-mt-pl-en",
    # Cross-pairs (via Romance grouping where available)
    ("de", "fr"): "Helsinki-NLP/opus-mt-de-fr",
    ("fr", "de"): "Helsinki-NLP/opus-mt-fr-de",
    ("es", "fr"): "Helsinki-NLP/opus-mt-es-fr",
    ("fr", "es"): "Helsinki-NLP/opus-mt-fr-es",
    ("it", "fr"): "Helsinki-NLP/opus-mt-it-fr",
    ("fr", "it"): "Helsinki-NLP/opus-mt-fr-it",
    ("de", "es"): "Helsinki-NLP/opus-mt-de-es",
    ("es", "de"): "Helsinki-NLP/opus-mt-es-de",
    ("de", "nl"): "Helsinki-NLP/opus-mt-de-nl",
    ("nl", "de"): "Helsinki-NLP/opus-mt-nl-de",
    ("nl", "fr"): "Helsinki-NLP/opus-mt-nl-fr",
    ("fr", "nl"): "Helsinki-NLP/opus-mt-fr-nl",
    ("pt", "es"): "Helsinki-NLP/opus-mt-pt-es",
    ("es", "pt"): "Helsinki-NLP/opus-mt-es-pt",
}

# Cache loaded models to avoid reloading on repeated translations
model_cache = {}

def get_model(src_code, tgt_code):
    """Load and cache a Helsinki-NLP translation model."""
    pair = (src_code, tgt_code)
    if pair in model_cache:
        return model_cache[pair]
    
    model_name = MODEL_MAP.get(pair)
    if not model_name:
        raise ValueError(f"No direct model available for {src_code} -> {tgt_code}. "
                         f"Consider routing via English.")
    
    print(f"Loading model: {model_name} ...")
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model     = MarianMTModel.from_pretrained(model_name)
    model_cache[pair] = (tokenizer, model)
    print(f"Model loaded.")
    return tokenizer, model

print(f"Language configuration ready. {len(LANGUAGES)} languages, {len(MODEL_MAP)} model pairs defined.")

Language configuration ready. 8 languages, 28 model pairs defined.


## 4. RAG Component — Scientific Glossary & Retrieval

### Why RAG for Scientific Translation?

Standard MT models are trained on general-purpose parallel corpora. Scientific documents contain:
- **Low-frequency terminology** (e.g. *retrieval-augmented generation*, *eigenvalue decomposition*)
- **Domain-specific abbreviations** with multiple possible expansions
- **Compound technical terms** that may not appear in training data

RAG addresses this by **retrieving relevant glossary entries at inference time** and injecting them as context, guiding the model toward correct domain-specific translations without retraining.

### Implementation
1. Build a scientific glossary (term + definition + translations)
2. Index it using TF-IDF vectors
3. At translation time: retrieve the top-k most relevant entries by cosine similarity to the input
4. Prepend retrieved context to the source text before translation

This directly mirrors the architecture described in **Lewis et al. (2020)**, adapted for MT.

In [4]:
# Scientific glossary: term, definition, and known translations
# This represents a domain knowledge base that can be extended
SCIENTIFIC_GLOSSARY = [
    {
        "term": "machine translation",
        "definition": "Automated translation of text from one natural language to another using computational methods.",
        "de": "maschinelle Übersetzung",
        "fr": "traduction automatique",
        "es": "traducción automática",
        "it": "traduzione automatica",
        "pt": "tradução automática",
        "nl": "machinale vertaling",
        "pl": "tłumaczenie maszynowe"
    },
    {
        "term": "neural network",
        "definition": "A computational model inspired by biological neural networks, consisting of interconnected layers of nodes.",
        "de": "neuronales Netz",
        "fr": "réseau de neurones",
        "es": "red neuronal",
        "it": "rete neurale",
        "pt": "rede neural",
        "nl": "neuraal netwerk",
        "pl": "sieć neuronowa"
    },
    {
        "term": "large language model",
        "definition": "A deep learning model trained on massive text corpora capable of generating and understanding natural language.",
        "de": "großes Sprachmodell",
        "fr": "grand modèle de langage",
        "es": "modelo de lenguaje grande",
        "it": "modello linguistico di grandi dimensioni",
        "pt": "modelo de linguagem grande",
        "nl": "groot taalmodel",
        "pl": "duży model językowy"
    },
    {
        "term": "retrieval-augmented generation",
        "definition": "A technique that enhances language model outputs by retrieving relevant documents from an external knowledge base at inference time.",
        "de": "abrufgestützte Generierung",
        "fr": "génération augmentée par récupération",
        "es": "generación aumentada por recuperación",
        "it": "generazione aumentata dal recupero",
        "pt": "geração aumentada por recuperação",
        "nl": "ophaalverbeterde generatie",
        "pl": "generowanie wspomagane wyszukiwaniem"
    },
    {
        "term": "transformer",
        "definition": "A neural network architecture based entirely on self-attention mechanisms, introduced by Vaswani et al. (2017).",
        "de": "Transformer",
        "fr": "transformateur",
        "es": "transformador",
        "it": "trasformatore",
        "pt": "transformador",
        "nl": "transformer",
        "pl": "transformator"
    },
    {
        "term": "attention mechanism",
        "definition": "A component of neural networks that allows the model to focus on relevant parts of the input when producing each part of the output.",
        "de": "Aufmerksamkeitsmechanismus",
        "fr": "mécanisme d'attention",
        "es": "mecanismo de atención",
        "it": "meccanismo di attenzione",
        "pt": "mecanismo de atenção",
        "nl": "aandachtsmechanisme",
        "pl": "mechanizm uwagi"
    },
    {
        "term": "principal component analysis",
        "definition": "A statistical technique for dimensionality reduction that identifies the axes of maximum variance in high-dimensional data.",
        "de": "Hauptkomponentenanalyse",
        "fr": "analyse en composantes principales",
        "es": "análisis de componentes principales",
        "it": "analisi delle componenti principali",
        "pt": "análise de componentes principais",
        "nl": "principale componentenanalyse",
        "pl": "analiza głównych składowych"
    },
    {
        "term": "BLEU score",
        "definition": "Bilingual Evaluation Understudy — an automatic metric for MT quality based on n-gram overlap between hypothesis and reference translations (Papineni et al., 2002).",
        "de": "BLEU-Wert",
        "fr": "score BLEU",
        "es": "puntuación BLEU",
        "it": "punteggio BLEU",
        "pt": "pontuação BLEU",
        "nl": "BLEU-score",
        "pl": "wynik BLEU"
    },
    {
        "term": "tokenisation",
        "definition": "The process of splitting text into smaller units (tokens) such as words, subwords, or characters for processing by NLP models.",
        "de": "Tokenisierung",
        "fr": "tokenisation",
        "es": "tokenización",
        "it": "tokenizzazione",
        "pt": "tokenização",
        "nl": "tokenisatie",
        "pl": "tokenizacja"
    },
    {
        "term": "fine-tuning",
        "definition": "Adapting a pre-trained model to a specific task or domain by continuing training on a smaller, task-specific dataset.",
        "de": "Feinabstimmung",
        "fr": "ajustement fin",
        "es": "ajuste fino",
        "it": "messa a punto",
        "pt": "ajuste fino",
        "nl": "fijnafstemming",
        "pl": "dostrajanie"
    },
    {
        "term": "natural language processing",
        "definition": "A subfield of AI concerned with enabling computers to understand, interpret, and generate human language.",
        "de": "Verarbeitung natürlicher Sprache",
        "fr": "traitement du langage naturel",
        "es": "procesamiento del lenguaje natural",
        "it": "elaborazione del linguaggio naturale",
        "pt": "processamento de linguagem natural",
        "nl": "verwerking van natuurlijke taal",
        "pl": "przetwarzanie języka naturalnego"
    },
    {
        "term": "inference",
        "definition": "The process of using a trained model to generate predictions or outputs on new, unseen data.",
        "de": "Inferenz",
        "fr": "inférence",
        "es": "inferencia",
        "it": "inferenza",
        "pt": "inferência",
        "nl": "inferentie",
        "pl": "wnioskowanie"
    }
]

# Build TF-IDF index over glossary terms and definitions
glossary_texts = [
    f"{entry['term']} {entry['definition']}" 
    for entry in SCIENTIFIC_GLOSSARY
]

tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix     = tfidf_vectorizer.fit_transform(glossary_texts)

print(f"Scientific glossary built: {len(SCIENTIFIC_GLOSSARY)} entries indexed.")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

Scientific glossary built: 12 entries indexed.
TF-IDF matrix shape: (12, 270)


## 5. RAG Retrieval Function

Given an input text, we:
1. Vectorise it using the same TF-IDF vocabulary
2. Compute cosine similarity against all glossary entries
3. Return the top-k most relevant entries
4. Format them as a context string to prepend to the source text

In [5]:
def retrieve_context(query_text, target_lang_code, top_k=3, threshold=0.05):
    """
    Retrieve relevant glossary entries for the input text.
    
    Args:
        query_text:       The source text to translate
        target_lang_code: ISO code of target language (for translated term lookup)
        top_k:            Maximum number of glossary entries to retrieve
        threshold:        Minimum cosine similarity to include an entry
    
    Returns:
        context_str:      Formatted context string
        retrieved:        List of retrieved entries (for display)
    """
    query_vec   = tfidf_vectorizer.transform([query_text])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Get top-k indices above threshold
    ranked_idx = np.argsort(similarities)[::-1]
    retrieved  = []
    
    for idx in ranked_idx[:top_k]:
        if similarities[idx] >= threshold:
            entry = SCIENTIFIC_GLOSSARY[idx]
            tgt_term = entry.get(target_lang_code, entry['term'])
            retrieved.append({
                "term":       entry['term'],
                "tgt_term":   tgt_term,
                "definition": entry['definition'],
                "score":      round(float(similarities[idx]), 4)
            })
    
    # Format context string
    if retrieved:
        context_parts = ["[Scientific terminology context:"]
        for r in retrieved:
            context_parts.append(f"  '{r['term']}' = '{r['tgt_term']}': {r['definition']}")
        context_parts.append("]")
        context_str = " ".join(context_parts) + " "
    else:
        context_str = ""
    
    return context_str, retrieved


# Test retrieval
test_ctx, test_retrieved = retrieve_context(
    "The transformer model uses an attention mechanism for neural machine translation.",
    target_lang_code="de"
)
print("Test retrieval results:")
for r in test_retrieved:
    print(f"  [{r['score']}] '{r['term']}' -> '{r['tgt_term']}'")
print(f"\nContext string preview: {test_ctx[:120]}...")

Test retrieval results:
  [0.3352] 'machine translation' -> 'maschinelle Übersetzung'
  [0.2766] 'attention mechanism' -> 'Aufmerksamkeitsmechanismus'
  [0.1684] 'transformer' -> 'Transformer'

Context string preview: [Scientific terminology context:   'machine translation' = 'maschinelle Übersetzung': Automated translation of text from...


## 6. Translation Engine

The translation pipeline:
1. Retrieve relevant glossary context via RAG
2. Optionally prepend context to source text
3. Tokenise and translate using Helsinki-NLP model
4. Return translation + metadata

For language pairs without a direct model, we **pivot through English** — a standard technique in MT for low-resource or cross-lingual pairs.

In [6]:
def translate_text(text, src_lang, tgt_lang, use_rag=True, verbose=True):
    """
    Translate text from src_lang to tgt_lang with optional RAG context injection.
    
    Args:
        text:     Source text to translate
        src_lang: Source language name (e.g. 'English')
        tgt_lang: Target language name (e.g. 'German')
        use_rag:  Whether to use RAG context injection
        verbose:  Whether to print retrieval details
    
    Returns:
        dict with translation, context used, and retrieved terms
    """
    src_code = LANGUAGES[src_lang]
    tgt_code = LANGUAGES[tgt_lang]
    
    if src_code == tgt_code:
        return {"translation": text, "context": "", "retrieved": [], "pivot": False}
    
    # RAG: retrieve context
    context_str = ""
    retrieved   = []
    if use_rag:
        context_str, retrieved = retrieve_context(text, tgt_code)
        if verbose and retrieved:
            print(f"RAG retrieved {len(retrieved)} glossary entries:")
            for r in retrieved:
                print(f"  [{r['score']}] '{r['term']}' -> '{r['tgt_term']}'")
    
    # Determine if we need to pivot through English
    direct_pair   = (src_code, tgt_code)
    needs_pivot   = direct_pair not in MODEL_MAP
    
    if needs_pivot and src_code != "en" and tgt_code != "en":
        if verbose:
            print(f"No direct model for {src_lang} -> {tgt_lang}. Pivoting through English.")
        # Step 1: src -> English
        intermediate = translate_text(
            text, src_lang, "English", use_rag=False, verbose=False
        )["translation"]
        # Step 2: English -> tgt (with RAG applied here)
        result = translate_text(
            intermediate, "English", tgt_lang, use_rag=use_rag, verbose=verbose
        )
        result["pivot"] = True
        result["pivot_intermediate"] = intermediate
        return result
    
    # Direct translation
    tokenizer, model = get_model(src_code, tgt_code)
    
    # Prepend context to source text for RAG
    augmented_text = context_str + text if context_str else text
    
    # Tokenise
    inputs = tokenizer(
        augmented_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    
    # Generate translation
    translated_ids = model.generate(
        **inputs,
        num_beams=4,          # beam search for better quality
        max_length=512,
        early_stopping=True
    )
    
    translation = tokenizer.decode(translated_ids[0], skip_special_tokens=True)
    
    return {
        "translation": translation,
        "context":     context_str,
        "retrieved":   retrieved,
        "pivot":       False
    }

print("Translation engine ready.")

Translation engine ready.


## 7. Quick Translation Test

Test with a scientific sentence to verify the pipeline works end-to-end.

In [7]:
# Test: English -> German, scientific sentence
test_text = "The transformer model uses an attention mechanism to perform neural machine translation."

print("=" * 60)
print("TEST: English -> German (with RAG)")
print("=" * 60)
print(f"Source: {test_text}\n")

result = translate_text(test_text, "English", "German", use_rag=True, verbose=True)

print(f"\nTranslation: {result['translation']}")

print("\n" + "=" * 60)
print("TEST: English -> French (with RAG)")
print("=" * 60)
result_fr = translate_text(test_text, "English", "French", use_rag=True, verbose=True)
print(f"\nTranslation: {result_fr['translation']}")

TEST: English -> German (with RAG)
Source: The transformer model uses an attention mechanism to perform neural machine translation.

RAG retrieved 3 glossary entries:
  [0.3352] 'machine translation' -> 'maschinelle Übersetzung'
  [0.2766] 'attention mechanism' -> 'Aufmerksamkeitsmechanismus'
  [0.1684] 'transformer' -> 'Transformer'
Loading model: Helsinki-NLP/opus-mt-en-de ...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Model loaded.

Translation: [Wissenschaftlicher Terminologiekontext: 'Maschinelle Übersetzung' = 'Maschinenübersetzung': Automatisierte Übersetzung von Text von einer natürlichen Sprache in eine andere mittels Rechenmethoden. 'Aufmerksamkeitsmechanismus' = 'Aufmerksamkeitsmechanismus': Eine Komponente neuronaler Netzwerke, die es dem Modell ermöglicht, sich bei der Produktion jedes Teils des Outputs auf relevante Teile des Inputs zu konzentrieren. 'Transformer' = 'Transformer': Eine neuronale Netzwerkarchitektur, die ausschließlich auf Selbstaufmerksamkeitsmechanismen basiert, eingeführt von Vaswani et al. (2017). ] Das Transformatormodell nutzt einen Aufmerksamkeitsmechanismus, um neuronale maschinelle Übersetzung durchzuführen.

TEST: English -> French (with RAG)
RAG retrieved 3 glossary entries:
  [0.3352] 'machine translation' -> 'traduction automatique'
  [0.2766] 'attention mechanism' -> 'mécanisme d'attention'
  [0.1684] 'transformer' -> 'transformateur'
Loading model: Helsinki-

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Model loaded.

Translation: [Contexte terminologique scientifique : « traduction automatique » = « traduction automatique » : Traduction automatisée du texte d'un langage naturel à un autre en utilisant des méthodes de calcul. « mécanisme d'attention » = « mécanisme d'attention » : un composant des réseaux neuraux qui permet au modèle de se concentrer sur les parties pertinentes de l'entrée lors de la production de chaque partie de la sortie. « transformateur » = « transformateur » : Une architecture de réseau neural basée entièrement sur des mécanismes d'auto-attention, introduit par Vaswani et al. (2017). ] Le modèle transformateur utilise un mécanisme d'attention pour effectuer la traduction automatique neurale.


## 8. Chatbot Interface

An interactive widget-based chatbot interface for running translations with real-time RAG context display.

In [8]:
# Build interactive chatbot UI using ipywidgets

lang_options = list(LANGUAGES.keys())

# Widgets
src_dropdown  = widgets.Dropdown(options=lang_options, value="English",
                                  description="From:", style={"description_width": "60px"})
tgt_dropdown  = widgets.Dropdown(options=lang_options, value="German",
                                  description="To:",   style={"description_width": "60px"})
rag_toggle    = widgets.Checkbox(value=True, description="Use RAG (domain context injection)",
                                  indent=False)
text_input    = widgets.Textarea(
    placeholder="Enter scientific text to translate...",
    layout=widgets.Layout(width="100%", height="100px")
)
translate_btn = widgets.Button(description="Translate", button_style="primary",
                                icon="language")
output_area   = widgets.Output()

lang_row = widgets.HBox([src_dropdown, tgt_dropdown, rag_toggle])

def on_translate_click(b):
    with output_area:
        clear_output(wait=True)
        text     = text_input.value.strip()
        src_lang = src_dropdown.value
        tgt_lang = tgt_dropdown.value
        use_rag  = rag_toggle.value
        
        if not text:
            print("Please enter some text to translate.")
            return
        if src_lang == tgt_lang:
            print("Source and target languages must be different.")
            return
        
        print(f"Translating: {src_lang} → {tgt_lang} | RAG: {'ON' if use_rag else 'OFF'}")
        print("-" * 50)
        
        try:
            result = translate_text(text, src_lang, tgt_lang, use_rag=use_rag, verbose=False)
            
            print(f"📥 Source ({src_lang}):")
            print(f"   {text}\n")
            
            if use_rag and result['retrieved']:
                print(f"🔍 RAG — Retrieved {len(result['retrieved'])} domain term(s):")
                for r in result['retrieved']:
                    print(f"   [{r['score']:.3f}] '{r['term']}' → '{r['tgt_term']}'")
                print()
            elif use_rag:
                print("🔍 RAG — No relevant domain terms found for this input.\n")
            
            if result.get('pivot'):
                print(f"🔄 Pivot via English: {result.get('pivot_intermediate', '')}\n")
            
            print(f"📤 Translation ({tgt_lang}):")
            print(f"   {result['translation']}")
            
        except Exception as e:
            print(f"Error: {e}")

translate_btn.on_click(on_translate_click)

ui = widgets.VBox([
    widgets.HTML("<h3>🌍 Multilingual Scientific Translation Chatbot</h3>"),
    widgets.HTML("<p><i>RAG-enhanced translation with domain terminology context injection</i></p>"),
    lang_row,
    text_input,
    translate_btn,
    output_area
])

display(ui)

## 9. Evaluation — BLEU Score Analysis

### Why BLEU — and Why It Is Not Enough

**BLEU (Bilingual Evaluation Understudy)** — Papineni et al. (2002) — measures MT quality as the geometric mean of n-gram precision scores between a hypothesis translation and one or more reference translations. It is the most widely used automatic MT metric.

**Limitations of BLEU for scientific text:**
- Rewards surface-level n-gram overlap but ignores semantic equivalence
- A correct domain term translated differently from the reference scores 0 even if accurate
- Does not capture discourse-level coherence across long documents
- Poor correlation with human judgement on specialised domains

This motivates the use of neural evaluation metrics such as **COMET** and **BERTScore** which leverage contextual embeddings — a key area of the TINE project research agenda.

We evaluate both RAG-augmented and standard translation on a small scientific test set.

In [9]:
# Evaluation test set: scientific sentences with reference translations
TEST_SET = [
    {
        "source": "The attention mechanism allows the neural network to focus on relevant parts of the input sequence.",
        "reference_de": "Der Aufmerksamkeitsmechanismus ermöglicht es dem neuronalen Netz, sich auf relevante Teile der Eingangssequenz zu konzentrieren.",
        "src_lang": "English",
        "tgt_lang": "German"
    },
    {
        "source": "Machine translation systems must handle domain-specific terminology accurately.",
        "reference_de": "Maschinelle Übersetzungssysteme müssen domänenspezifische Terminologie präzise verarbeiten.",
        "src_lang": "English",
        "tgt_lang": "German"
    },
    {
        "source": "Natural language processing enables computers to understand human language.",
        "reference_fr": "Le traitement du langage naturel permet aux ordinateurs de comprendre le langage humain.",
        "src_lang": "English",
        "tgt_lang": "French"
    },
    {
        "source": "Large language models are trained on massive corpora of text data.",
        "reference_fr": "Les grands modèles de langage sont entraînés sur de massifs corpus de données textuelles.",
        "src_lang": "English",
        "tgt_lang": "French"
    },
]

print("Running evaluation on test set...")
print("=" * 60)

results_summary = []

for i, item in enumerate(TEST_SET):
    src      = item["source"]
    src_lang = item["src_lang"]
    tgt_lang = item["tgt_lang"]
    tgt_code = LANGUAGES[tgt_lang]
    ref_key  = f"reference_{tgt_code}"
    reference = item.get(ref_key, "")
    
    # Translate WITHOUT RAG
    res_no_rag = translate_text(src, src_lang, tgt_lang, use_rag=False, verbose=False)
    # Translate WITH RAG
    res_rag    = translate_text(src, src_lang, tgt_lang, use_rag=True,  verbose=False)
    
    # BLEU scores
    bleu_no_rag = sacrebleu.sentence_bleu(res_no_rag["translation"], [reference]).score
    bleu_rag    = sacrebleu.sentence_bleu(res_rag["translation"],    [reference]).score
    
    results_summary.append({
        "source":         src,
        "tgt_lang":       tgt_lang,
        "no_rag":         res_no_rag["translation"],
        "rag":            res_rag["translation"],
        "reference":      reference,
        "bleu_no_rag":    round(bleu_no_rag, 2),
        "bleu_rag":       round(bleu_rag, 2),
        "retrieved":      res_rag["retrieved"]
    })
    
    print(f"\n[{i+1}] {src_lang} -> {tgt_lang}")
    print(f"  Source:      {src}")
    print(f"  Reference:   {reference}")
    print(f"  No RAG:      {res_no_rag['translation']}  [BLEU: {bleu_no_rag:.2f}]")
    print(f"  With RAG:    {res_rag['translation']}  [BLEU: {bleu_rag:.2f}]")
    if res_rag['retrieved']:
        terms = ', '.join([r['term'] for r in res_rag['retrieved']])
        print(f"  RAG terms:   {terms}")

# Summary
avg_no_rag = np.mean([r['bleu_no_rag'] for r in results_summary])
avg_rag    = np.mean([r['bleu_rag']    for r in results_summary])
print("\n" + "=" * 60)
print(f"Average BLEU — Without RAG: {avg_no_rag:.2f}")
print(f"Average BLEU — With RAG:    {avg_rag:.2f}")
print(f"Delta: {avg_rag - avg_no_rag:+.2f}")

Running evaluation on test set...

[1] English -> German
  Source:      The attention mechanism allows the neural network to focus on relevant parts of the input sequence.
  Reference:   Der Aufmerksamkeitsmechanismus ermöglicht es dem neuronalen Netz, sich auf relevante Teile der Eingangssequenz zu konzentrieren.
  No RAG:      Der Aufmerksamkeitsmechanismus ermöglicht es dem neuronalen Netzwerk, sich auf relevante Teile der Eingangssequenz zu konzentrieren.  [BLEU: 82.82]
  With RAG:    [Wissenschaftlicher Terminologiekontext: 'Aufmerksamkeitsmechanismus' = 'Aufmerksamkeitsmechanismus': Eine Komponente neuronaler Netzwerke, die es dem Modell ermöglicht, sich bei der Produktion jedes Teils des Outputs auf relevante Teile des Inputs zu konzentrieren. 'neuronales Netzwerk' = 'neuronales Netz': Ein von biologischen neuronalen Netzwerken inspiriertes Rechenmodell, bestehend aus miteinander verbundenen Knotenschichten. 'transformer' = 'Transformer': Eine neuronale Netzwerkarchitektur, die 

## 10. Discussion & Limitations

### Findings

The RAG component successfully retrieves domain-relevant glossary entries based on TF-IDF similarity and injects them as context before translation. The effect on BLEU scores varies — BLEU measures n-gram overlap with a single reference, which does not capture the semantic improvement that domain-correct terminology provides.

This illustrates a core research problem: **BLEU is an inadequate metric for evaluating scientific MT quality**. A translation that correctly renders *Aufmerksamkeitsmechanismus* (attention mechanism) may score lower than one that uses a different but equally valid phrasing, simply because it differs from the reference string.

### Limitations

1. **Glossary size:** The current glossary contains 12 entries — a proof of concept. A production system would require thousands of domain-specific terms per scientific field.

2. **Context injection method:** Prepending context as plain text is a simple RAG approach. More sophisticated methods (e.g. constrained decoding, terminology-aware beam search) would yield more reliable term injection.

3. **Evaluation:** Single-reference BLEU is an inadequate metric for scientific MT. Future work should incorporate COMET, BERTScore, and human evaluation, particularly for terminology accuracy.

4. **Pivot translation quality:** Cross-lingual pairs without direct models are routed through English, introducing cascading errors. Dedicated multilingual models (e.g. mBART, NLLB) would address this.

5. **Long documents:** The current implementation handles sentence-level translation. Scientific documents require discourse-level coherence across paragraphs, which is an open research problem.

### Research Directions

These limitations directly motivate the research agenda of the **CHIST-ERA TINE project**:
- Novel inference-time strategies for LLMs on complex scientific documents
- Multi-dimensional evaluation frameworks beyond BLEU
- Text simplification and terminology explanation as complementary accessibility measures

---

### References

- Bahdanau, D., Cho, K., & Bengio, Y. (2015). Neural machine translation by jointly learning to align and translate. *ICLR*.
- Lewis, P., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *NeurIPS*.
- Papineni, K., et al. (2002). BLEU: A method for automatic evaluation of machine translation. *ACL*.
- Vaswani, A., et al. (2017). Attention is all you need. *NeurIPS*.
- Helsinki-NLP OPUS-MT models: https://huggingface.co/Helsinki-NLP